In [ ]:
from pathlib import Path
from urllib.parse import urlparse
from typing_extensions import TypedDict

import requests
from langgraph.graph import StateGraph, START, END

c:\Users\user\anaconda3\envs\compute\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.3.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [ ]:
DOWNLOAD_DIR = Path("download_xml")


class XmlState(TypedDict, total=False):
    sitemap_url: str
    xml_file_path: str


In [ ]:
def get_site_name(url: str) -> str:
    netloc = urlparse(url).netloc.lower()

    if netloc.startswith("www."):
        netloc = netloc[4:]

    return netloc.split(".")[0]


def download_xml(state: XmlState) -> dict:
    DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

    site_name = get_site_name(state["sitemap_url"])
    file_path = DOWNLOAD_DIR / f"{site_name}.xml"

    response = requests.get(state["sitemap_url"], timeout=30)
    response.raise_for_status()

    file_path.write_text(response.text, encoding="utf-8")

    return {"xml_file_path": str(file_path)}


In [ ]:
builder = StateGraph(XmlState)
builder.add_node("download_xml", download_xml)
builder.add_edge(START, "download_xml")
builder.add_edge("download_xml", END)

graph = builder.compile()

if __name__ == "__main__":
    result = graph.invoke(
        {
            "sitemap_url": "https://www.motortrend.com/sitemap.xml"
        }
    )

    print(result["xml_file_path"])

download_xml\motortrend.xml


In [16]:
from pathlib import Path
from trafilatura import fetch_url, extract

url = "https://www.automotiveworld.com/news/daimler-buses-breaks-ground-on-new-rome-service-centre/"

downloaded = fetch_url(url)

if not downloaded:
    raise ValueError("Failed to download the webpage with Trafilatura.")

# LLM-friendly markdown
markdown = extract(
    downloaded,
    url=url,
    output_format="markdown",
    with_metadata=True,
    include_tables=True,
    include_images=True,
    include_links=True,
    favor_recall=True,
    deduplicate=True,
)

output_dir = Path("page_outputs")
output_dir.mkdir(exist_ok=True)

markdown_file = output_dir / "article.md"
markdown_file.write_text(markdown or "", encoding="utf-8")

print("Markdown saved to:", markdown_file)

Markdown saved to: page_outputs\article.md
